# Enterprise Knowledge Navigator
## DAY 2: HyDE + Multi-Query + Hybrid Search + Groq LLM
### Full RAG pipeline giving real answers with page citations

**What we build today:**
- Reload everything from Day 1 (ChromaDB + BM25 + chunks)
- HyDE: generate hypothetical answer then search with its embedding
- Multi-query: 3 query variants merged for higher recall
- Hybrid Search: BM25 + Vector combined with RRF score fusion
- Cross-encoder reranking for precise top-5 selection
- Full RAG pipeline: question to answer with page citations
- Conversational memory: 10-turn window with summary compression


## STEP 1: Mount Drive and Reload Day 1 Data

Everything from Day 1 is saved in your Drive.
We reload it here so Day 2 starts instantly without re-processing.
This is why persistent storage matters in real production systems.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted!")


In [ ]:
!pip install -q sentence-transformers chromadb rank-bm25 groq
print("Libraries ready!")


In [ ]:
import os, json, pickle, re, time
from datetime import datetime
import numpy as np
from tqdm import tqdm
import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from groq import Groq
import torch

BASE = "/content/drive/MyDrive/EnterpriseKnowledgeNavigator"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Imports OK! Device: " + device)


In [ ]:
# Reload ChromaDB
chroma_client = chromadb.PersistentClient(path=BASE + "/chromadb")
collection = chroma_client.get_collection("tesla_knowledge_base")
print("ChromaDB loaded! Vectors: " + str(collection.count()))

# Reload BM25
with open(BASE + "/bm25_index/bm25_index.pkl", "rb") as f:
    bm25_data = pickle.load(f)
bm25 = bm25_data["index"]
all_chunks = bm25_data["chunks"]
print("BM25 loaded! Chunks: " + str(len(all_chunks)))

# Reload embedding model
print("Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print("All Day 1 data reloaded!")


## STEP 2: Setup Groq (Free LLM API)

**Get your free Groq API key:**
1. Go to console.groq.com
2. Sign up with Google account
3. Click API Keys in left sidebar
4. Create new key and copy it
5. Paste it below replacing YOUR_GROQ_API_KEY_HERE

Groq runs Llama 3 70B for free. Faster than OpenAI. No cost.


In [ ]:
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"

groq_client = Groq(api_key=GROQ_API_KEY)

test = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say exactly: Groq is working!"}],
    max_tokens=20
)
print("Groq test: " + test.choices[0].message.content)


## STEP 3: Core Helper Functions

Three functions used throughout Day 2:
- embed_text: converts any text to a 384-dim embedding vector
- call_groq: sends prompt to Llama 3 70B and returns response
- tokenize_bm25: prepares text for keyword search


In [ ]:
def embed_text(text):
    return embedding_model.encode(text, normalize_embeddings=True)

def call_groq(prompt, system="You are a helpful Tesla company expert.", max_tokens=500):
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=0.1
    )
    return response.choices[0].message.content

def tokenize_bm25(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 -]", " ", text)
    return [t for t in text.split() if len(t) > 1]

print("Helper functions ready!")


## STEP 4: Baseline Vector Search

Before building advanced retrieval we establish a baseline.
This shows how naive RAG works and why we need to improve it.
You will see the clear difference when we add HyDE and multi-query.


In [ ]:
def vector_search(query, n=10):
    qe = embed_text(query).tolist()
    results = collection.query(
        query_embeddings=[qe],
        n_results=n,
        include=["documents", "metadatas", "distances"]
    )
    chunks = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        chunks.append({"text": doc, "metadata": meta, "score": round(1 - dist, 4)})
    return chunks

test_q = "What is Tesla revenue in 2023?"
baseline = vector_search(test_q, n=3)
print("Baseline Vector Search for: " + test_q)
for r in baseline:
    print("  Score: " + str(r["score"]) + " | " + r["metadata"]["source"] + " p" + str(r["metadata"]["page_number"]))
    print("  " + r["text"][:150] + "...")
    print()


## STEP 5: BM25 Keyword Search

BM25 scores chunks based on exact keyword frequency.
Notice it finds DIFFERENT results than vector search for the same query.
This difference is exactly why we combine them in hybrid search.


In [ ]:
def bm25_search(query, n=10):
    tokens = tokenize_bm25(query)
    scores = bm25.get_scores(tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n]
    chunks = []
    for idx in top_idx:
        if scores[idx] > 0:
            chunks.append({
                "text": all_chunks[idx]["text"],
                "metadata": all_chunks[idx]["metadata"],
                "score": round(float(scores[idx]), 4)
            })
    return chunks

bm25_res = bm25_search(test_q, n=3)
print("BM25 Search for: " + test_q)
for r in bm25_res:
    print("  Score: " + str(r["score"]) + " | " + r["metadata"]["source"] + " p" + str(r["metadata"]["page_number"]))
    print("  " + r["text"][:150] + "...")
    print()
print("Notice: BM25 and Vector return DIFFERENT results for same query.")
print("This is exactly why hybrid search beats either alone.")


## STEP 6: Hybrid Search with RRF Fusion

**What is RRF? Reciprocal Rank Fusion**

RRF combines BM25 and vector search rankings into one score.
Formula: RRF score = 1 divided by (rank + 60) for each result list.

Example:
Chunk A: rank 1 in vector + rank 1 in BM25 = 1/61 + 1/61 = highest score
Chunk B: rank 1 in vector + not in BM25   = 1/61 + 0    = medium score
Chunk C: rank 50 in both                  = 1/110 + 1/110 = low score

The number 60 prevents top ranks from dominating too much.
This formula was proven in academic research to outperform weighted averaging.


In [ ]:
def hybrid_search(query, n_final=10, n_retrieve=20):
    vec_results = vector_search(query, n=n_retrieve)
    bm25_results = bm25_search(query, n=n_retrieve)

    rrf_scores = {}
    chunk_data = {}

    for rank, result in enumerate(vec_results):
        cid = result["metadata"]["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1.0 / (rank + 60)
        chunk_data[cid] = result

    for rank, result in enumerate(bm25_results):
        cid = result["metadata"]["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1.0 / (rank + 60)
        chunk_data[cid] = result

    sorted_ids = sorted(rrf_scores.keys(), key=lambda k: rrf_scores[k], reverse=True)
    results = []
    for cid in sorted_ids[:n_final]:
        item = dict(chunk_data[cid])
        item["rrf_score"] = round(rrf_scores[cid], 6)
        results.append(item)
    return results

hybrid_res = hybrid_search(test_q, n_final=3)
print("Hybrid RRF Search for: " + test_q)
for r in hybrid_res:
    print("  RRF: " + str(r["rrf_score"]) + " | " + r["metadata"]["source"] + " p" + str(r["metadata"]["page_number"]))
    print("  " + r["text"][:150] + "...")
    print()


## STEP 7: HyDE - Hypothetical Document Embeddings

**Why HyDE improves RAG:**

Problem: a short user query embedding may not match dense factual document text.
The query "What is Tesla revenue?" is short and vague.
Document chunks contain rich sentences like "Tesla reported 96.8 billion in revenue for fiscal 2023."

HyDE solution:
Step 1: Ask LLM to write a hypothetical answer paragraph for the question.
Step 2: Embed the hypothetical answer instead of the short question.
Step 3: Search with that richer embedding.

Why it works: a hypothetical answer looks like actual document text.
Embedding a fake detailed answer finds the real answer because they are semantically similar.


In [ ]:
def hyde_search(query, n=10):
    hyde_prompt = (
        "Write a short factual paragraph answering this question about Tesla.\n"
        "Question: " + query + "\n"
        "Write only the answer paragraph, no introduction."
    )
    hypothetical = call_groq(hyde_prompt, max_tokens=150)
    print("HyDE hypothetical answer:")
    print("  " + hypothetical[:200])
    print()

    hyde_emb = embed_text(hypothetical).tolist()
    results = collection.query(
        query_embeddings=[hyde_emb],
        n_results=n,
        include=["documents", "metadatas", "distances"]
    )
    chunks = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        chunks.append({"text": doc, "metadata": meta, "score": round(1 - dist, 4)})
    return chunks

print("=" * 50)
print("Testing HyDE Retrieval")
print("=" * 50)
hyde_res = hyde_search("What are Tesla main products?", n=3)
for r in hyde_res:
    print("  Score: " + str(r["score"]) + " | " + r["metadata"]["source"] + " p" + str(r["metadata"]["page_number"]))
    print("  " + r["text"][:150] + "...")
    print()


## STEP 8: Multi-Query Retrieval

**Why multi-query helps:**

A single query may miss relevant chunks because of wording differences.
"Tesla revenue" may miss chunks that say "Tesla earnings" or "Tesla sales figures".

Solution: generate 3 different versions of the same question.
Search with all 3 versions and merge results with deduplication.
This increases recall significantly - finding chunks one query would miss.


In [ ]:
def multi_query_search(query, n_per_query=8):
    variant_prompt = (
        "Generate 3 different search queries to find information about this question.\n"
        "Original question: " + query + "\n"
        "Return exactly 3 queries, one per line, no numbering, no explanation."
    )
    variants_text = call_groq(variant_prompt, max_tokens=100)
    variants = [q.strip() for q in variants_text.strip().split("\n") if len(q.strip()) > 5][:3]
    all_variants = [query] + variants

    print("Multi-query variants:")
    for v in all_variants:
        print("  - " + v)
    print()

    seen = {}
    for variant in all_variants:
        results = vector_search(variant, n=n_per_query)
        for r in results:
            cid = r["metadata"]["chunk_id"]
            if cid not in seen or r["score"] > seen[cid]["score"]:
                seen[cid] = r

    sorted_results = sorted(seen.values(), key=lambda x: x["score"], reverse=True)
    print("Unique chunks across all variants: " + str(len(sorted_results)))
    return sorted_results

print("=" * 50)
print("Testing Multi-Query Retrieval")
print("=" * 50)
mq_res = multi_query_search("How does Tesla make money?")
print("Top 3:")
for r in mq_res[:3]:
    print("  " + r["metadata"]["source"] + " p" + str(r["metadata"]["page_number"]) + " score=" + str(r["score"]))
    print("  " + r["text"][:150] + "...")
    print()


## STEP 9: Cross-Encoder Reranking

**Two-stage retrieval: why it matters**

Stage 1 retrieval: fast but approximate. Gets 20 candidates quickly.
Stage 2 reranking: slow but precise. Picks best 5 from the 20.

Bi-encoder (what we used so far): encodes query and document separately.
Cross-encoder: looks at query AND document together. Much more accurate.

Cross-encoder is too slow to use on all 275 chunks.
So we use it only on the top 20 candidates from Stage 1.
This two-stage approach is what production search systems use.


In [ ]:
print("Loading cross-encoder reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)
print("Reranker loaded!")

def rerank_chunks(query, chunks, top_n=5):
    if not chunks:
        return []
    pairs = [(query, chunk["text"]) for chunk in chunks]
    scores = reranker.predict(pairs)
    for i, chunk in enumerate(chunks):
        chunk["rerank_score"] = round(float(scores[i]), 4)
    reranked = sorted(chunks, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_n]

print("Testing reranker...")
candidates = hybrid_search("Tesla battery technology", n_final=15)
reranked = rerank_chunks("Tesla battery technology", candidates, top_n=3)
print("Top 3 after reranking:")
for r in reranked:
    print("  Rerank: " + str(r["rerank_score"]) + " | " + r["metadata"]["source"])
    print("  " + r["text"][:150] + "...")
    print()


## STEP 10: Complete RAG Pipeline

**The full pipeline combining every technique:**
1. User asks a question
2. HyDE generates hypothetical answer and embeds it
3. Multi-query generates 3 variants and retrieves candidates
4. Hybrid search adds BM25 + vector candidates
5. All candidates merged and deduplicated
6. Cross-encoder reranks to find best 5
7. Groq Llama 3 generates answer with source citations

This is enterprise-grade RAG. Most tutorials only do steps 1 and 7.


In [ ]:
def full_rag_pipeline(query, use_hyde=True, use_multiquery=True):
    print("Query: " + query)
    print("-" * 50)

    all_candidates = {}

    if use_hyde:
        print("Running HyDE...")
        for r in hyde_search(query, n=10):
            cid = r["metadata"]["chunk_id"]
            if cid not in all_candidates:
                all_candidates[cid] = r

    if use_multiquery:
        print("Running multi-query...")
        for r in multi_query_search(query, n_per_query=8):
            cid = r["metadata"]["chunk_id"]
            if cid not in all_candidates:
                all_candidates[cid] = r

    print("Running hybrid search...")
    for r in hybrid_search(query, n_final=15):
        cid = r["metadata"]["chunk_id"]
        if cid not in all_candidates:
            all_candidates[cid] = r

    candidates = list(all_candidates.values())
    print("Total unique candidates: " + str(len(candidates)))

    print("Reranking...")
    top_chunks = rerank_chunks(query, candidates, top_n=5)

    context_parts = []
    sources = []
    for i, chunk in enumerate(top_chunks):
        label = "[Source " + str(i+1) + ": " + chunk["metadata"]["source"] + " page " + str(chunk["metadata"]["page_number"]) + "]"
        context_parts.append(label + "\n" + chunk["text"])
        sources.append(label)

    context = "\n\n".join(context_parts)

    print("Generating answer with Groq Llama 3...")
    answer_prompt = (
        "Answer the question using ONLY the context below.\n"
        "Cite sources like [Source 1] when using information.\n"
        "If context does not contain the answer, say so clearly.\n\n"
        "Context:\n" + context + "\n\n"
        "Question: " + query + "\n\nAnswer:"
    )
    answer = call_groq(answer_prompt, max_tokens=400)

    print("\n" + "=" * 50)
    print("ANSWER:")
    print(answer)
    print("\nSOURCES:")
    for s in sources:
        print("  " + s)
    print("=" * 50)

    return {"query": query, "answer": answer, "sources": sources, "chunks": top_chunks}

# TEST THE FULL PIPELINE
result = full_rag_pipeline("What is Tesla revenue in 2023 and how many vehicles were delivered?")


## STEP 11: Conversational Memory (10-turn window)

**Why conversational memory matters:**

Without memory: every question is independent and context is lost.
User asks: What is Tesla revenue? -> answer given
User asks: How does that compare to 2022? -> LLM has no idea what "that" refers to!

With 10-turn memory: last 10 questions and answers stay in context.
LLM knows what was discussed and handles follow-up questions naturally.

Summary compression: when history exceeds 10 turns, compress old turns.
Old turns summarized into 3 sentences to keep context size manageable.


In [ ]:
class ConversationalRAG:
    def __init__(self, max_turns=10):
        self.max_turns = max_turns
        self.history = []
        self.summary = ""

    def compress_history(self):
        old = self.history[:5]
        self.history = self.history[5:]
        turns_text = ""
        for t in old:
            turns_text += "Q: " + t["question"] + "\nA: " + t["answer"] + "\n\n"
        new_summary = call_groq(
            "Summarize these conversation turns in 3 sentences keeping key facts:\n\n" + turns_text,
            max_tokens=150
        )
        self.summary = (self.summary + " " + new_summary).strip()
        print("History compressed.")

    def build_history_ctx(self):
        parts = []
        if self.summary:
            parts.append("Previous summary: " + self.summary)
        for t in self.history[-5:]:
            parts.append("User: " + t["question"])
            parts.append("Assistant: " + t["answer"])
        return "\n".join(parts)

    def chat(self, question):
        if len(self.history) >= self.max_turns:
            self.compress_history()

        candidates = hybrid_search(question, n_final=15)
        top_chunks = rerank_chunks(question, candidates, top_n=4)

        context_parts = []
        sources = []
        for i, chunk in enumerate(top_chunks):
            label = "[Source " + str(i+1) + ": " + chunk["metadata"]["source"] + " p" + str(chunk["metadata"]["page_number"]) + "]"
            context_parts.append(label + "\n" + chunk["text"])
            sources.append(label)

        context = "\n\n".join(context_parts)
        history_ctx = self.build_history_ctx()

        prompt = "You are a Tesla expert. Answer using context and history. Cite sources like [Source 1].\n\n"
        if history_ctx:
            prompt += "Conversation history:\n" + history_ctx + "\n\n"
        prompt += "Context:\n" + context + "\n\nQuestion: " + question + "\n\nAnswer:"

        answer = call_groq(prompt, max_tokens=350)
        self.history.append({"question": question, "answer": answer})

        return {"answer": answer, "sources": sources, "turn": len(self.history)}


print("=" * 50)
print("TESTING CONVERSATIONAL RAG")
print("=" * 50)

rag_chat = ConversationalRAG(max_turns=10)

questions = [
    "What is Tesla revenue in 2023?",
    "How does that compare to 2022?",
    "What products drive most of that revenue?",
]

for q in questions:
    print("\nUSER: " + q)
    resp = rag_chat.chat(q)
    print("ASSISTANT: " + resp["answer"])
    print("Sources: " + str(resp["sources"]))
    print("Turn: " + str(resp["turn"]) + "/" + str(rag_chat.max_turns))


## STEP 12: Save for Day 3


In [ ]:
with open(BASE + "/data/processed/rag_chat.pkl", "wb") as f:
    pickle.dump(rag_chat, f)

summary_d2 = {
    "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "components_built": [
        "vector_search",
        "bm25_search",
        "hybrid_search_rrf",
        "hyde_retrieval",
        "multi_query",
        "cross_encoder_reranking",
        "full_rag_pipeline",
        "conversational_memory_10turn"
    ],
    "llm": "Groq Llama 3 70B free",
    "status": "DAY 2 COMPLETE"
}

with open(BASE + "/outputs/day2_summary.json", "w") as f:
    json.dump(summary_d2, f, indent=2)

print("=" * 50)
print("DAY 2 COMPLETE!")
print("=" * 50)
print(json.dumps(summary_d2, indent=2))
print("\nFile > Download > Download .ipynb")
print("Commit: [Day-15] Enterprise Knowledge Navigator - Day2 RAG Pipeline")


## DAY 2 COMPLETE!

### What you built:
- Baseline vector search and BM25 keyword search
- Hybrid Search with RRF score fusion
- HyDE: search with a hypothetical answer embedding
- Multi-query: 3 query variants merged for higher recall
- Cross-encoder reranking: precise top-5 from wide candidates
- Full RAG pipeline combining all techniques
- Conversational memory with 10-turn window and summary compression

### Key concepts learned:
1. RRF: 1 divided by rank plus 60 combines rankings without bias
2. HyDE: searching with a fake answer finds the real answer
3. Multi-query: different phrasings find different relevant chunks
4. Two-stage retrieval: wide net then precise reranking
5. Memory compression: summarize old turns to keep context manageable

### Day 3 preview:
- Neo4j Graph RAG: extract entities and build knowledge graph
- FastAPI backend: REST API endpoints for the full pipeline
- Streamlit frontend: chat UI with source highlighting
- RAGAS evaluation dashboard
